# LEAN-LLM-OPT audit-and-repair fallback

Reusable fallback definitions. Append the shared cell and one backend cell after a Primary notebook's definition cells. `LEAN_LLM_OPT_4.1_Large-scale_with_Fallback.ipynb` and `LEAN_LLM_OPT_gpt_oss_20b_Large-scale_with_Fallback.ipynb` in this folder show the complete notebooks.

## Audit-and-repair fallback: shared definitions

The fallback runs after the Primary workflow has produced a model, code and solution. It audits the formulation logic, the data usage and the code in that order, repairs the first failing layer, re-executes the program in an isolated worker that may read only the registered CSV files, and audits again. A repaired candidate replaces the Primary output only if all three layers pass and the solve is optimal; otherwise the Primary output is kept. Reference objectives are never used.

In [ ]:
from __future__ import annotations

import ast as fb_ast
import contextlib as fb_contextlib
import copy as fb_copy
import hashlib as fb_hashlib
import io as fb_io
import json as fb_json
import math as fb_math
import os as fb_os
import re as fb_re
import subprocess as fb_subprocess
import sys as fb_sys
import time as fb_time
from pathlib import Path as FBPath
from typing import Literal as FBLiteral

from pydantic import BaseModel as FBBaseModel, ConfigDict as FBConfigDict, Field as FBField

FB_ROOT = FBPath(PROJECT_ROOT).resolve()
FB_PRIMARY_MODEL = MODEL_SNAPSHOT
FB_MAX_REPAIRS = 3
FB_MAX_AUDITS = 4


def fb_sha(path):
    digest = fb_hashlib.sha256()
    with FBPath(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def fb_canonical(value):
    if isinstance(value, FBBaseModel):
        value = value.model_dump(mode="json")
    return fb_json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"), default=str)


def fb_hash(value):
    return fb_hashlib.sha256(fb_canonical(value).encode("utf-8")).hexdigest()


def fb_atomic_json(path, value):
    path = FBPath(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(fb_json.dumps(value, ensure_ascii=False, sort_keys=True, indent=2, default=str) + "\n", encoding="utf-8")
    fb_os.replace(temp, path)


def fb_load_json(path):
    return fb_json.loads(FBPath(path).read_text(encoding="utf-8"))


class FallbackStrict(FBBaseModel):
    model_config = FBConfigDict(extra="forbid")


class FallbackFileScope(FallbackStrict):
    file_index: int = FBField(ge=0)
    mode: FBLiteral["ALL_ROWS", "FILTER_ROWS", "UNRESOLVED"]
    column: str
    operator: FBLiteral["NONE", "PREFIX", "SUFFIX", "CONTAINS", "EQUALS"]
    value: str
    query_evidence: str
    explanation: str


class FallbackScope(FallbackStrict):
    files: list[FallbackFileScope]


class FallbackLayer(FallbackStrict):
    status: FBLiteral["PASS", "FAIL"]
    error_codes: list[str]
    evidence: list[str]


class FallbackAudit(FallbackStrict):
    logic: FallbackLayer
    data: FallbackLayer
    code: FallbackLayer
    first_failed_layer: FBLiteral["logic", "data", "code", "none"]


class FallbackFormulationRepair(FallbackStrict):
    mathematical_formulation: str
    repair_summary: str


class FallbackCodeRepair(FallbackStrict):
    python_code: str
    repair_summary: str


def fb_profile(paths, query):
    profile = []
    compact_query = str(query).casefold()
    for index, path in enumerate(paths):
        frame = _read_csv(path)
        columns = []
        for name in frame.columns:
            series = frame[name].astype(str)
            matching = [value for value in series.unique() if value.strip().casefold() in compact_query]
            columns.append({"name": str(name), "examples": (matching[:6] + series.head(3).tolist())[:9]})
        profile.append({"file_index": index, "row_count": len(frame), "columns": columns})
    return profile


def fb_expected_rows(path, decision, query):
    frame = _read_csv(path)
    if decision.mode == "ALL_ROWS":
        return [int(i) for i in frame.index]
    if decision.mode == "UNRESOLVED":
        raise RuntimeError("SCOPE_UNRESOLVED")
    if decision.column not in frame.columns:
        raise ValueError("SCOPE_COLUMN_INVALID")
    if not decision.value.strip() or decision.value.casefold() not in str(query).casefold():
        raise ValueError("SCOPE_VALUE_NOT_IN_QUERY")
    if not decision.query_evidence or decision.query_evidence not in str(query):
        raise ValueError("SCOPE_EVIDENCE_NOT_EXACT_QUERY_SPAN")
    values = frame[decision.column].astype(str).str.strip().str.casefold()
    target = decision.value.strip().casefold()
    if decision.operator == "PREFIX":
        selected = values.str.startswith(target)
    elif decision.operator == "SUFFIX":
        selected = values.str.endswith(target)
    elif decision.operator == "CONTAINS":
        selected = values.str.contains(target, regex=False)
    elif decision.operator == "EQUALS":
        selected = values.eq(target)
    else:
        raise ValueError("SCOPE_OPERATOR_INVALID")
    result = [int(i) for i in frame.index[selected]]
    if not result:
        raise ValueError("SCOPE_EMPTY_SELECTION")
    return result


def fb_prepare_native_observation(payload, paths, query):
    """Apply the Primary's source-label, unit and logical-table preparation to a corrected payload."""
    if FB_PRIMARY_MODEL != "gpt-oss:20b" or "_annotate_logical_tables" not in globals():
        return payload
    tables = _load_tables("\n".join(str(path) for path in paths))
    _validate_named_source_coverage(payload, tables, query)
    payload = _annotate_logical_tables(payload, tables)
    if _relational_export_layout(tables):
        payload, _report = _uview_prepare(payload, query)
        payload = _annotate_logical_tables(payload, tables)
        payload = _logical_records_view(payload)
    return payload


def fb_validate_planned_observation(parsed, paths, query):
    """Prove physical rows, each view's order, and exact original value lineage.

    Views may overlap and physical files may be split across views. Coverage is
    the union of physical source rows; duplicate rows inside one view are invalid.
    A prepared unit view is replayed with the unmodified native source functions.
    """
    from collections import defaultdict
    if not isinstance(parsed, dict) or not isinstance(parsed.get("tables"), list):
        raise ValueError("PLANNED_OBSERVATION_INVALID")
    frames = [_read_csv(path) for path in paths]
    physical, views, identities = defaultdict(set), [], set()
    raw = fb_copy.deepcopy(parsed)
    prepared = "unit_preparation" in parsed
    for table, raw_table in zip(parsed["tables"], raw["tables"]):
        index = table.get("file_index")
        if not isinstance(index, int) or isinstance(index, bool) or not 0 <= index < len(paths):
            raise ValueError("PLANNED_UNREGISTERED_FILE")
        frame = frames[index]
        table_id = table.get("table_id")
        if not isinstance(table_id, str) or not table_id or table_id in identities:
            raise ValueError("PLANNED_DUPLICATE_OR_INVALID_TABLE_ID")
        identities.add(table_id)
        columns = table.get("columns")
        if not isinstance(columns, list) or not columns or len(columns) != len(set(columns)):
            raise ValueError("PLANNED_COLUMNS_INVALID")
        records = table.get("records")
        if not isinstance(records, list):
            raise ValueError("PLANNED_RECORDS_INVALID")
        row_ids = []
        for record, raw_record in zip(records, raw_table["records"]):
            row_id = record.get("source_row")
            if not isinstance(row_id, int) or isinstance(row_id, bool) or row_id not in frame.index:
                raise ValueError("PLANNED_SOURCE_ROW_INVALID")
            row_ids.append(row_id)
            values = record.get("values")
            if values is None:
                raise ValueError("PLANNED_SOURCE_VALUES_MISSING")
            if not isinstance(values, dict) or not values:
                raise ValueError("PLANNED_SOURCE_VALUES_INVALID")
            if any(column not in frame.columns for column in values):
                raise ValueError("PLANNED_SOURCE_COLUMN_INVALID")
            if table.get("columns") is not None and set(table["columns"]) != set(values):
                raise ValueError("PLANNED_RECORD_COLUMN_MISMATCH")
            original = {str(column): str(frame.loc[row_id, column]) for column in values}
            restored = dict(values)
            if "source_unit_fields" in record:
                if not prepared or set(record["source_unit_fields"]) != {"amount", "unit"}:
                    raise ValueError("PLANNED_SOURCE_UNIT_PROVENANCE_INVALID")
                restored.update(record["source_unit_fields"])
            if restored != original:
                raise ValueError("PLANNED_ALTERED_SOURCE_VALUES")
            if prepared:
                digest = fb_hashlib.sha256(fb_json.dumps(original, ensure_ascii=False, sort_keys=True).encode()).hexdigest()
                if record.get("source_values_sha256") != digest:
                    raise ValueError("PLANNED_SOURCE_VALUES_HASH_INVALID")
                raw_record["values"] = original
                raw_record.pop("source_unit_fields", None)
                raw_record.pop("source_values_sha256", None)
        if row_ids != sorted(set(row_ids)):
            raise ValueError("PLANNED_VIEW_ORDER_OR_DUPLICATE_ROW")
        if table.get("returned_rows", len(records)) != len(records):
            raise ValueError("PLANNED_RETURNED_ROW_COUNT_INVALID")
        if table.get("original_rows", len(frame)) != len(frame):
            raise ValueError("PLANNED_ORIGINAL_ROW_COUNT_INVALID")
        physical[index].update(row_ids)
        views.append({"file_index": index, "table_id": table_id, "source_rows": row_ids})
    if any(key in parsed for key in ("logical_table_index", "logical_records", "unit_preparation")):
        if FB_PRIMARY_MODEL != "gpt-oss:20b":
            raise ValueError("PLANNED_UNSUPPORTED_DERIVED_VIEW")
        for key in ("unit_preparation", "logical_table_index", "logical_table_evidence",
                    "logical_records", "logical_records_metadata"):
            raw.pop(key, None)
        replay = fb_prepare_native_observation(raw, paths, query)
        for key in ("tables", "unit_preparation", "logical_table_index", "logical_table_evidence",
                    "logical_records", "logical_records_metadata"):
            if parsed.get(key) != replay.get(key):
                raise ValueError("PLANNED_DERIVED_PROVENANCE_MISMATCH:" + key)
    return {index: sorted(rows) for index, rows in physical.items()}, views


def fb_compare_retrieval(paths, scope, observation, status, route, query):
    """Use each Primary's route mode and actual source-row evidence."""
    from collections import defaultdict, deque
    by_index = {item.file_index: item for item in scope.files}
    if len(by_index) != len(scope.files) or set(by_index) != set(range(len(paths))):
        raise ValueError("SCOPE_FILE_INDEX_MISMATCH")
    planned = CSVQA_MODE_BY_ROUTE.get(route) == "planned" and status in {"PLANNED", "FALLBACK_FULL_DATA"}
    legacy_source = status in {"LEGACY_VALIDATED_SOURCE_ROWS", "LEGACY_SOURCE_VALIDATION_FALLBACK"}
    parsed = None
    legacy_actual = defaultdict(list)
    if planned:
        parsed = fb_json.loads(observation)
        try:
            physical_rows, planned_views = fb_validate_planned_observation(parsed, paths, query)
            provenance_errors = []
        except (ValueError, KeyError, TypeError) as exc:
            physical_rows, planned_views = {}, []
            provenance_errors = [f"{type(exc).__name__}:{exc}"]
    elif legacy_source:
        # A validated row array can still be incomplete. Match values and their
        # multiplicity to the registered sources; never infer coverage from status.
        rows = fb_json.loads(observation)
        if not isinstance(rows, list):
            raise ValueError("LEGACY_SOURCE_ARRAY_REQUIRED")
        available = {}
        for index, path in enumerate(paths):
            frame = _read_csv(path)
            signatures = defaultdict(deque)
            for row_id, row in frame.iterrows():
                key = tuple(sorted((str(column), str(row[column])) for column in frame.columns))
                signatures[key].append(int(row_id))
            available[str(FBPath(path).resolve())] = (index, signatures)
        for row in rows:
            if not isinstance(row, dict) or not isinstance(row.get("values"), dict):
                raise ValueError("LEGACY_SOURCE_ROW_INVALID")
            source = str(FBPath(str(row.get("source", ""))).expanduser().resolve())
            if source not in available:
                raise ValueError("LEGACY_UNREGISTERED_SOURCE")
            index, signatures = available[source]
            key = tuple(sorted((str(column), str(value)) for column, value in row["values"].items()))
            if not signatures[key]:
                raise ValueError("LEGACY_ALTERED_OR_DUPLICATE_ROW")
            legacy_actual[index].append(signatures[key].popleft())
    files = []
    for index, path in enumerate(paths):
        decision = by_index[index]
        if decision.mode == "UNRESOLVED":
            files.append({"file_index": index, "mode": "UNRESOLVED", "coverage_ok": False,
                          "expected_count": None, "actual_count": None, "expected_sha": None, "actual_sha": None})
            continue
        expected = fb_expected_rows(path, decision, query)
        if planned:
            actual = physical_rows.get(index, [])
            ok = actual == expected and not provenance_errors
            evidence = "planned_physical_union_and_view_order"
        elif legacy_source:
            actual = legacy_actual[index]
            ok = set(expected).issubset(set(actual))
            evidence = "legacy_verified_source_rows"
        elif route in {"TP", "NRM", "RA", "FLP", "AP"} and status in {"LEGACY_FULL_DATA", "LEGACY_FAISS"} and observation:
            actual = [int(i) for i in _read_csv(path).index]
            ok = set(expected).issubset(set(actual))
            evidence = "legacy_full_context_coverage"
        elif route == "Others" and status == "LEGACY_SCHEMA" and observation:
            actual, ok = None, None
            evidence = "schema_preview_runtime_read_required"
        else:
            actual, ok = [], False
            evidence = "retrieval_not_observed"
        files.append({"file_index": index, "mode": decision.mode, "operator": decision.operator,
                      "column": decision.column, "value": decision.value, "evidence": evidence,
                      "coverage_ok": ok, "expected_count": len(expected),
                      "actual_count": len(actual) if actual is not None else None,
                      "expected_sha": fb_hash(expected), "actual_sha": fb_hash(actual) if actual is not None else None,
                      "missing_preview": [i for i in expected if actual is not None and i not in set(actual)][:12]})
    return {"files": files, "unresolved": any(item.mode == "UNRESOLVED" for item in scope.files),
            "coverage_failed": any(item["coverage_ok"] is False for item in files),
            "all_proven": all(item["coverage_ok"] is True for item in files),
            "planned_views": planned_views if planned else [],
            "provenance_errors": provenance_errors if planned else []}


def fb_corrected_observation(paths, scope, query, route, observation=""):
    decisions = {item.file_index: item for item in scope.files}
    if len(decisions) != len(scope.files) or set(decisions) != set(range(len(paths))):
        raise ValueError("SCOPE_FILE_INDEX_MISMATCH")
    if CSVQA_MODE_BY_ROUTE.get(route) == "planned":
        frames = [_read_csv(path) for path in paths]
        expected = {i: fb_expected_rows(path, decisions[i], query) for i, path in enumerate(paths)}
        try:
            old = fb_json.loads(observation) if observation else {}
        except (ValueError, TypeError):
            old = {}
        old = old if isinstance(old, dict) else {}
        tables, covered, table_ids = [], {i: set() for i in range(len(paths))}, set()
        for table in old.get("tables", []):
            index = table.get("file_index")
            table_id = table.get("table_id")
            if not isinstance(index, int) or index not in expected or not isinstance(table_id, str) or not table_id or table_id in table_ids:
                continue
            frame = frames[index]
            columns = table.get("columns") or list(frame.columns)
            if not isinstance(columns, list) or not columns or any(column not in frame.columns for column in columns):
                columns = list(frame.columns)
            ids = sorted({record.get("source_row") for record in table.get("records", [])
                          if isinstance(record, dict) and isinstance(record.get("source_row"), int)
                          and record["source_row"] in set(expected[index])})
            if not ids:
                continue
            fixed = {key: fb_copy.deepcopy(value) for key, value in table.items() if key != "records"}
            fixed.update(columns=columns, original_rows=len(frame), returned_rows=len(ids),
                         records=[{"source_row": row_id, "values": {str(column): str(frame.loc[row_id, column])
                                   for column in columns}} for row_id in ids])
            tables.append(fixed)
            table_ids.add(table_id)
            covered[index].update(ids)
        for index, path in enumerate(paths):
            frame = frames[index]
            ids = [row for row in expected[index] if row not in covered[index]]
            if not ids and covered[index]:
                continue
            view_index = 0
            while f"file_{index}_view_{view_index}" in table_ids:
                view_index += 1
            table_id = f"file_{index}_view_{view_index}"
            table_ids.add(table_id)
            tables.append({"table_id": table_id, "file_index": index,
                           "file_name": FBPath(path).name, "role": "data", "columns": list(frame.columns),
                           "original_rows": len(frame), "returned_rows": len(ids),
                           "filters": {"logic": "and", "conditions": []},
                           "records": [{"source_row": row_id,
                                        "values": {str(column): str(frame.loc[row_id, column]) for column in frame.columns}}
                                       for row_id in ids]})
        payload = {"route": route, "query": query, "tables": tables,
                   "relationships": fb_copy.deepcopy(old.get("relationships", [])), "ignored_file_indices": [],
                   "validation": {"status": "POST_PRIMARY_CORRECTED"}}
        payload = fb_prepare_native_observation(payload, paths, query)
        return fb_json.dumps(payload, ensure_ascii=False, sort_keys=True)
    if FB_PRIMARY_MODEL != "gpt-oss:20b":
        rows = []
        for index, path in enumerate(paths):
            frame = _read_csv(path)
            for row_id in fb_expected_rows(path, decisions[index], query):
                rows.append({"source": str(path),
                             "values": {str(column): str(frame.loc[row_id, column]) for column in frame.columns}})
        return fb_json.dumps(rows, ensure_ascii=False)
    raise ValueError("LEGACY_SOURCE_REPAIR_UNSUPPORTED_FOR_NATIVE_OSS")


class FallbackAuditService:
    """Model-neutral scope, audit, and targeted-repair prompts."""
    def parse(self, kind, schema, system, payload, layer="", max_tokens=12000):
        raise NotImplementedError("Use a backend service defined below")

    def scope(self, query, paths):
        system = """After the complete Primary, judge the exact input rows each CSV should contribute. Return one decision per file_index. ALL_ROWS unless the query explicitly restricts records. Optimization choices are not input filters. For FILTER_ROWS copy a literal value and exact evidence span from the query; specify column and PREFIX, SUFFIX, CONTAINS or EQUALS. Trim and casefold matching; preserve duplicate rows. If the exact condition cannot be represented, use UNRESOLVED. Do not use file paths, IDs, labels or expected objective."""
        return self.parse("retrieval_scope", FallbackScope, system,
                          {"query": query, "files": fb_profile(paths, query)}, max_tokens=6000)

    def audit(self, query, candidate, comparison):
        system = """Audit the latest artifact in Logic -> Data -> Code order. Logic: objective, variable domain, full constraints, query meaning. Data: required rows/columns and faithful data binding. Code: implementation and runtime. Use the supplied csvqa_mode and retrieval evidence. Planned evidence, including planned Others, requires physical source_row union coverage, source order within each table view, faithful source values/provenance, and complete runtime CSVQA_DATA. Different views may overlap; do not count overlap as duplicated physical input. Source-tagged legacy evidence must cover the required rows; a required subset covered by a full context is not a retrieval failure, but inspect whether the formulation/code apply the query filter. Only a legacy schema-preview artifact requires complete CSV reads at runtime. Local coverage_failed must make Data FAIL. The supplied runtime error must make Code FAIL. Choose the first failing layer, give concrete error codes and evidence. Never use Gold or expected objective."""
        return self.parse("artifact_audit", FallbackAudit, system,
                          {"query": query, "route": candidate["route"], "retrieval": comparison,
                            "csvqa_mode": candidate.get("csvqa_mode") or CSVQA_MODE_BY_ROUTE.get(candidate["route"]),
                           "formulation": candidate["formulation"], "observation": candidate["observation"][:100000],
                           "code": candidate["code"], "runtime": candidate["runtime"]}, max_tokens=10000)

    def formulation_repair(self, layer, query, candidate, audit):
        system = f"""Repair only the first failed {layer} layer. Return a complete corrected mathematical formulation, not Python. Preserve all correct parts. Use the query and supplied observation. The original Primary code generator will then regenerate Python. Never use labels or expected objective."""
        return self.parse("repair", FallbackFormulationRepair, system,
                          {"query": query, "route": candidate["route"], "layer": layer,
                           "formulation": candidate["formulation"], "observation": candidate["observation"][:180000],
                           "audit": audit.model_dump(mode="json")}, layer=layer, max_tokens=16000)

    def code_repair(self, query, candidate, audit):
        system = """Repair only Python code. Preserve the mathematical formulation, data scope and route. Return complete executable Gurobi source without fences. Read only registered CSVs, optimize once, leave solved model in m or model. Do not write files, use network or subprocess, or dynamically execute code. Resolve the listed runtime error."""
        return self.parse("repair", FallbackCodeRepair, system,
                          {"query": query, "route": candidate["route"],
                           "formulation": candidate["formulation"], "observation": candidate["observation"][:180000],
                           "code": candidate["code"], "runtime": candidate["runtime"],
                           "audit": audit.model_dump(mode="json")}, layer="code", max_tokens=22000)


def fb_force_local(audit, comparison, runtime):
    layers = {name: getattr(audit, name).model_copy(deep=True) for name in ("logic", "data", "code")}
    if comparison.get("unresolved"):
        layers["data"].status = "FAIL"
        layers["data"].error_codes.append("SCOPE_UNRESOLVED")
    elif comparison.get("coverage_failed"):
        layers["data"].status = "FAIL"
        layers["data"].error_codes.append("RETRIEVAL_ROW_MISMATCH")
    if runtime.get("error") or runtime.get("objective") is None:
        layers["code"].status = "FAIL"
        layers["code"].error_codes.append("EXECUTION_FAILED")
    first = next((name for name in ("logic", "data", "code") if layers[name].status == "FAIL"), "none")
    return FallbackAudit(**layers, first_failed_layer=first)


FB_BAD_IMPORTS = {"os", "sys", "subprocess", "socket", "requests", "urllib", "pathlib", "shutil", "pickle", "importlib", "http", "ftplib"}
FB_BAD_CALLS = {"open", "eval", "exec", "compile", "__import__", "input", "breakpoint"}
FB_BAD_ATTRS = {"system", "popen", "run", "call", "check_call", "check_output", "to_csv", "to_excel", "to_pickle", "write_text", "write_bytes", "unlink", "remove", "rmdir", "mkdir", "dump", "dumps", "loadtxt", "genfromtxt", "read_pickle"}


def fb_validate_fallback_code(code):
    source = _source_candidate(code)
    try:
        tree = fb_ast.parse(source)
    except SyntaxError as exc:
        return [f"SYNTAX_ERROR:{exc.msg}"]
    issues = []
    for node in fb_ast.walk(tree):
        if isinstance(node, (fb_ast.Import, fb_ast.ImportFrom)):
            names = [alias.name for alias in node.names] if isinstance(node, fb_ast.Import) else [node.module or ""]
            if any(name.split(".")[0] in FB_BAD_IMPORTS for name in names):
                issues.append("FORBIDDEN_IMPORT")
        if isinstance(node, fb_ast.Call):
            fn = node.func
            if isinstance(fn, fb_ast.Name) and fn.id in FB_BAD_CALLS:
                issues.append("FORBIDDEN_CALL")
            if isinstance(fn, fb_ast.Attribute) and fn.attr in FB_BAD_ATTRS:
                issues.append("FORBIDDEN_ATTRIBUTE_CALL")
        if isinstance(node, fb_ast.Attribute) and node.attr.startswith("__"):
            issues.append("DUNDER_ATTRIBUTE")
    return sorted(set(issues))


FB_WORKER = r'''
import ast, builtins, contextlib, io, json, os, pathlib, re, sys
import gurobipy as gp
import pandas as pd
payload = json.loads(sys.stdin.read())
registered = {str(pathlib.Path(p).resolve()) for p in payload["paths"]}
original_read_csv = pd.read_csv
def registered_read_csv(path, *args, **kwargs):
    if not isinstance(path, (str, pathlib.Path)) or str(pathlib.Path(path).resolve()) not in registered:
        raise PermissionError("UNREGISTERED_CSV")
    return original_read_csv(path, *args, **kwargs)
pd.read_csv = registered_read_csv
out = {"objective": None, "error": None, "status": "FAILED"}
stream = io.StringIO()
try:
    with contextlib.redirect_stdout(stream):
        native = payload.get("native_execution_source")
        if native:
            native_namespace = {"__name__": "__main__", "gp": gp, "ast": ast, "os": os, "re": re}
            exec(native, native_namespace)
            objective, solution = native_namespace["execute_code"](payload["code"])
            if isinstance(objective, str):
                out.update(status=objective.upper(), error="SOLVER_NOT_OPTIMAL:" + objective.upper())
            else:
                out.update(objective=float(objective), status="OPTIMAL")
            out["execution_policy"] = "native_primary_execute_code"
        else:
            namespace = {"__name__": "__main__"}
            exec(payload["code"], namespace)
            model = namespace["m"] if "m" in namespace else namespace["model"]
            if model.Status != gp.GRB.OPTIMAL:
                raise RuntimeError("SOLVER_NOT_OPTIMAL:" + str(model.Status))
            out.update(objective=float(model.ObjVal), status="OPTIMAL")
except Exception as exc:
    out["error"] = type(exc).__name__ + ": " + str(exc)
    if getattr(exc, "code", "") in {"MODEL_INTERFACE_MISSING", "MODEL_INTERFACE_AMBIGUOUS", "MODEL_INTERFACE_DISPOSED"}:
        out["status"] = exc.code
out["log_tail"] = stream.getvalue()[-4000:]
print(json.dumps(out, ensure_ascii=False))
'''


def fb_native_execution_source():
    """Reuse the Primary notebook's own execution functions in the isolated worker."""
    if FB_PRIMARY_MODEL != "gpt-oss:20b":
        return ""
    notebook = fb_load_json(NOTEBOOK_PATH)
    required = {"_source_candidate", "ModelInterfaceError", "_resolve_generated_model", "execute_code"}
    found = {}
    for cell in notebook["cells"]:
        if cell.get("cell_type") != "code":
            continue
        source = "".join(cell.get("source", []))
        for node in fb_ast.parse(source).body:
            if isinstance(node, (fb_ast.FunctionDef, fb_ast.ClassDef)) and node.name in required:
                if node.name in found:
                    raise RuntimeError("Duplicate native execution definition: " + node.name)
                found[node.name] = fb_ast.get_source_segment(source, node)
    if set(found) != required:
        raise RuntimeError("Native execution contract definitions changed")
    return "\n\n".join(found[name] for name in ("_source_candidate", "ModelInterfaceError", "_resolve_generated_model", "execute_code"))


def fb_execute_fallback(code, paths, timeout=240):
    source = _source_candidate(code)
    issues = fb_validate_fallback_code(source)
    if issues:
        return {"objective": None, "error": ",".join(issues), "status": "STATIC_REJECTED", "log_tail": ""}
    try:
        process = fb_subprocess.run(
            [fb_sys.executable, "-c", FB_WORKER],
            input=fb_json.dumps({"code": source, "paths": list(map(str, paths)),
                                 "native_execution_source": fb_native_execution_source()}),
            text=True, capture_output=True, cwd=FB_ROOT, timeout=timeout, check=False,
        )
        if process.returncode != 0:
            return {"objective": None, "error": f"WORKER_EXIT:{process.returncode}",
                    "status": "FAILED", "log_tail": process.stderr[-4000:]}
        result = fb_json.loads(process.stdout.strip())
        return result
    except fb_subprocess.TimeoutExpired:
        return {"objective": None, "error": "WORKER_TIMEOUT", "status": "FAILED", "log_tail": ""}
    except Exception as exc:
        return {"objective": None, "error": f"WORKER_ERROR:{type(exc).__name__}:{exc}",
                "status": "FAILED", "log_tail": ""}


def fb_primary_case(case):
    query, address = case["query"], case["dataset_address"]
    payload = {"problem_id": case["case_id"], "Query": query, "dataset_address": address}
    output = fb_io.StringIO()
    try:
        with fb_contextlib.redirect_stdout(output):
            record = execute_pipeline_case(payload)
        error = None
    except Exception as exc:
        record = getattr(exc, "pipeline_context", {"problem_id": case["case_id"]})
        error = f"{type(exc).__name__}: {exc}"
    record = dict(record)
    record["primary_error"] = error
    record["primary_log_tail"] = output.getvalue()[-4000:]
    return record


def fb_candidate(primary):
    route = primary.get("assigned_route") or "Others"
    return {"route": route, "formulation": primary.get("generated_model", ""),
            "observation": primary.get("csvqa_observation", ""),
            "csvqa_status": primary.get("csvqa_status"), "csvqa_mode": primary.get("csvqa_mode"),
            "code": primary.get("solve_code", ""),
            "runtime": {"objective": primary.get("final_objective") if primary.get("final_ok") else None,
                        "error": primary.get("primary_error"),
                        "status": "OPTIMAL" if primary.get("final_ok") else "FAILED",
                        "log_tail": primary.get("primary_log_tail", "")}}


def fb_generate_primary_code(formulation, route, query, payload="", observation="", has_data=True):
    """Generate code with the Primary's own code generator and data binding."""
    if not has_data:
        return get_code(formulation, route, original_query=query)
    return get_csv_code(formulation, route, query, data_payload=payload,
                        legacy_observation=observation if CSVQA_MODE_BY_ROUTE.get(route) == "legacy" else "")


def fb_bind_authoritative_data(code, candidate):
    """Replace model-written bindings with the complete observed runtime data."""
    source = _source_candidate(code)
    observation = candidate.get("observation", "")
    mode = candidate.get("csvqa_mode") or CSVQA_MODE_BY_ROUTE.get(candidate.get("route"))
    bindings = {}
    if mode == "planned" and observation:
        bindings["CSVQA_DATA"] = fb_json.loads(observation)
    elif candidate.get("csvqa_status") in {"LEGACY_VALIDATED_SOURCE_ROWS", "LEGACY_SOURCE_VALIDATION_FALLBACK"} and observation:
        bindings["LEGACY_OBSERVATION"] = observation
        bindings["LEGACY_RECORDS"] = _legacy_records_from_observation(observation)
    if not bindings:
        return source
    tree = fb_ast.parse(source)
    kept = []
    for node in tree.body:
        if isinstance(node, (fb_ast.Assign, fb_ast.AnnAssign)):
            targets = node.targets if isinstance(node, fb_ast.Assign) else [node.target]
            if targets and all(isinstance(target, fb_ast.Name) and target.id in bindings for target in targets):
                continue
        kept.append(node)
    tree.body = kept
    for node in fb_ast.walk(tree):
        if isinstance(node, fb_ast.Name) and isinstance(node.ctx, (fb_ast.Store, fb_ast.Del)) and node.id in bindings:
            raise ValueError("AUTHORITATIVE_DATA_REBIND:" + node.id)
        if isinstance(node, (fb_ast.Subscript, fb_ast.Attribute)) and isinstance(node.ctx, (fb_ast.Store, fb_ast.Del)):
            root = node
            while isinstance(root, (fb_ast.Subscript, fb_ast.Attribute)):
                root = root.value
            if isinstance(root, fb_ast.Name) and root.id in bindings:
                raise ValueError("AUTHORITATIVE_DATA_MUTATION:" + root.id)
    prefix = "\n".join(f"{name} = {pprint.pformat(value, sort_dicts=False, width=120)}" for name, value in bindings.items())
    # Future imports must remain before the injected ordinary assignments.
    future = [node for node in tree.body if isinstance(node, fb_ast.ImportFrom) and node.module == "__future__"]
    tree.body = [node for node in tree.body if node not in future]
    return "\n".join([*(fb_ast.unparse(node) for node in future), prefix, fb_ast.unparse(tree)])


def fb_generate_legacy_schema_code(case, formulation, observation):
    """Reuse the native Others code template, holding the repaired plan fixed."""
    original_chain, original_preview = globals()["LLMChain"], globals()["csv_schema_preview"]
    def repaired_chain(*args, **kwargs):
        prompt = kwargs.get("prompt")
        if prompt is not None and "abstract_plan" not in prompt.input_variables:
            class FixedFormulation:
                def run(self, **unused):
                    return formulation
            return FixedFormulation()
        return original_chain(*args, **kwargs)
    globals()["LLMChain"] = repaired_chain
    globals()["csv_schema_preview"] = lambda _address, query="": observation
    try:
        return _source_candidate(get_Others_response(case["query"], case["dataset_address"])["code"])
    finally:
        globals()["LLMChain"], globals()["csv_schema_preview"] = original_chain, original_preview


def fb_regenerate_from_observation(case, route, observation):
    query, address = case["query"], case["dataset_address"]
    planned = CSVQA_MODE_BY_ROUTE.get(route) == "planned"
    if route == "Others" and not planned:
        original_preview = globals()["csv_schema_preview"]
        globals()["csv_schema_preview"] = lambda _address, query="": observation
        try:
            result = get_Others_response(query, address)
        finally:
            globals()["csv_schema_preview"] = original_preview
        formulation, code = result["formulation"], _source_candidate(result["code"])
        status = "LEGACY_SCHEMA"
    else:
        original_builder = globals()["build_csvqa_components"]
        status = "PLANNED" if planned else ("LEGACY_FULL_DATA" if FB_PRIMARY_MODEL == "gpt-oss:20b" else "LEGACY_VALIDATED_SOURCE_ROWS")
        def corrected_builder(_address, _system, tool_description, *, route, user_query="", result_sink=None):
            state = result_sink if result_sink is not None else {}
            state.update({"observation": observation,
                          "trace": {"status": status, "payload_hash": fb_hashlib.sha256(observation.encode()).hexdigest()}})
            return make_llm(), Tool(name="CSVQA", func=lambda _request: observation,
                                    description=tool_description), state
        globals()["build_csvqa_components"] = corrected_builder
        try:
            result = invoke_formulation(route, query, address)
        finally:
            globals()["build_csvqa_components"] = original_builder
        formulation = result["formulation"]
        payload = observation if planned else ""
        code = _source_candidate(fb_generate_primary_code(formulation, route, query, payload, observation))
        code = fb_bind_authoritative_data(code, {"route": route, "observation": observation,
                                           "csvqa_mode": "planned" if planned else "legacy", "csvqa_status": status})
    return {"route": route, "formulation": formulation, "observation": observation,
            "csvqa_status": status, "csvqa_mode": "planned" if planned else "legacy",
            "code": code, "runtime": fb_execute_fallback(code, [item["path"] for item in case["paths"]])}


def fb_regenerate_code(case, candidate, formulation):
    route, query = candidate["route"], case["query"]
    if route == "Others" and candidate.get("csvqa_status") == "LEGACY_SCHEMA":
        code = fb_generate_legacy_schema_code(case, formulation, candidate["observation"])
    else:
        payload = candidate["observation"] if CSVQA_MODE_BY_ROUTE.get(route) == "planned" else ""
        code = fb_generate_primary_code(formulation, route, query, payload, candidate["observation"], bool(case["paths"]))
    code = fb_bind_authoritative_data(code, candidate)
    candidate = fb_copy.deepcopy(candidate)
    candidate["formulation"] = formulation
    candidate["code"] = code
    candidate["runtime"] = fb_execute_fallback(code, [item["path"] for item in case["paths"]])
    return candidate


def fb_solve_case(case, run_root, *, resume=True, service=None, primary=None):
    directory = FBPath(run_root) / "cases" / case["case_id"]
    final_path = directory / "final_result.json"
    fingerprint = fb_hash({"case_id": case["case_id"], "query": case["query"], "paths": case["paths"]})
    if resume and final_path.exists():
        old = fb_load_json(final_path)
        if old.get("fingerprint") != fingerprint:
            raise RuntimeError(f"Incompatible completed checkpoint: {case['case_id']}")
        return old
    directory.mkdir(parents=True, exist_ok=True)
    started = fb_time.perf_counter()
    if primary is None:
        primary = fb_primary_case(case)
    else:
        if primary.get("problem_id", case["case_id"]) != case["case_id"]:
            raise ValueError("Supplied Primary belongs to a different case")
        primary = fb_copy.deepcopy(primary)
    fb_atomic_json(directory / "primary.json", primary)
    candidate = fb_candidate(primary)
    original = fb_copy.deepcopy(candidate)
    service = service or GPT41FallbackService()
    paths = [FBPath(item["path"]) for item in case["paths"]]
    audits, repairs, scope, warning = [], [], None, ""
    used = set()
    validated = False
    comparison = {"files": [], "unresolved": True, "coverage_failed": False, "all_proven": False}
    try:
        scope = service.scope(case["query"], paths)
        comparison = fb_compare_retrieval(paths, scope, candidate["observation"],
                                          candidate["csvqa_status"], candidate["route"], case["query"])
    except Exception as exc:
        warning = f"SCOPE_AUDIT_FAILED:{type(exc).__name__}:{exc}"
    fb_atomic_json(directory / "scope.json", {"scope": scope.model_dump(mode="json") if scope else None,
                                               "comparison": comparison, "warning": warning})
    if not warning:
        for index in range(FB_MAX_AUDITS):
            try:
                audit = fb_force_local(service.audit(case["query"], candidate, comparison),
                                       comparison, candidate["runtime"])
            except Exception as exc:
                warning = f"AUDIT_FAILED:{type(exc).__name__}:{exc}"
                break
            audits.append(audit.model_dump(mode="json"))
            fb_atomic_json(directory / f"audit_{index:02d}.json", audits[-1])
            layer = audit.first_failed_layer
            if layer == "none":
                validated = True
                break
            if comparison["unresolved"] or layer in used or len(repairs) >= FB_MAX_REPAIRS:
                warning = f"UNRESOLVED_OR_REPEATED_LAYER:{layer}"
                break
            used.add(layer)
            before = {key: fb_hash(candidate[key]) for key in ("formulation", "observation", "code")}
            try:
                if layer == "data" and comparison["coverage_failed"]:
                    observation = fb_corrected_observation(paths, scope, case["query"], candidate["route"], candidate["observation"])
                    candidate = fb_regenerate_from_observation(case, candidate["route"], observation)
                    comparison = fb_compare_retrieval(paths, scope, candidate["observation"],
                                                      candidate["csvqa_status"], candidate["route"], case["query"])
                    summary = f"Corrected source rows; {FB_PRIMARY_MODEL} regenerated formulation and code."
                elif layer in {"logic", "data"}:
                    repair = service.formulation_repair(layer, case["query"], candidate, audit)
                    candidate = fb_regenerate_code(case, candidate, repair.mathematical_formulation)
                    summary = repair.repair_summary
                else:
                    repair = service.code_repair(case["query"], candidate, audit)
                    candidate["code"] = fb_bind_authoritative_data(repair.python_code, candidate)
                    candidate["runtime"] = fb_execute_fallback(candidate["code"], paths)
                    summary = repair.repair_summary
            except Exception as exc:
                warning = f"REPAIR_FAILED:{layer}:{type(exc).__name__}:{exc}"
                break
            after = {key: fb_hash(candidate[key]) for key in ("formulation", "observation", "code")}
            repairs.append({"layer": layer, "summary": summary, "before": before, "after": after})
            fb_atomic_json(directory / f"repair_{len(repairs):02d}_{layer}.json", repairs[-1])
    if validated:
        selected = candidate
        source = "COMPOSITE" if repairs else "PRIMARY"
        trust = "FALLBACK_VALIDATED" if repairs else "PRIMARY_VALIDATED"
    elif original["runtime"]["objective"] is not None:
        selected, source, trust = original, "PRIMARY", "PRIMARY_UNVERIFIED"
    else:
        selected = fb_copy.deepcopy(candidate)
        selected["runtime"] = {**selected["runtime"], "objective": None, "status": "FAILED",
                               "error": selected["runtime"].get("error") or warning or "FALLBACK_NOT_VALIDATED"}
        source, trust = "NONE", "FAILED"
    result = {"case_id": case["case_id"], "fingerprint": fingerprint,
              "query_sha": fb_hash(case["query"]), "paths": case["paths"], "primary": primary,
              "candidate": candidate, "selected": {"source": source, "trust": trust, **selected},
              "scope": scope.model_dump(mode="json") if scope else None,
              "comparison": comparison, "audits": audits, "repairs": repairs,
              "fallback_calls": service.calls,
              "warning": warning,
              "elapsed_seconds": round(fb_time.perf_counter() - started, 3)}
    fb_atomic_json(final_path, result)
    return result


## GPT-4.1 fallback backend

Scope, audit and repair requests use GPT-4.1 with structured outputs. A data-scope check compares the rows the query requires with the rows the Primary retrieved. Each layer is repaired at most once; logic and data repairs regenerate code with the Primary's own code generator.

In [ ]:
FALLBACK_MODEL = "gpt-4.1-2025-04-14"


class GPT41FallbackService(FallbackAuditService):
    """Scope, audit and repair requests through the OpenAI structured-output API."""

    def __init__(self, client=None, model=FALLBACK_MODEL):
        if client is None:
            from openai import OpenAI
            client = OpenAI(timeout=240.0, max_retries=2)
        self.client, self.model, self.calls = client, model, []

    def parse(self, kind, schema, system, payload, layer="", max_tokens=12000):
        started = fb_time.perf_counter()
        record = {"kind": kind, "layer": layer, "model": self.model, "status": "STARTED"}
        self.calls.append(record)
        try:
            completion = self.client.chat.completions.parse(
                model=self.model,
                messages=[{"role": "system", "content": system},
                          {"role": "user", "content": fb_canonical(payload)}],
                response_format=schema, temperature=0.0, max_completion_tokens=max_tokens)
            usage = completion.usage
            record.update(input_tokens=int(usage.prompt_tokens or 0) if usage else None,
                          output_tokens=int(usage.completion_tokens or 0) if usage else None)
            message = completion.choices[0].message
            if message.refusal:
                raise RuntimeError("Audit model refused structured output")
            if message.parsed is None:
                raise RuntimeError("Audit model returned no structured output")
            record["status"] = "COMPLETED"
            return message.parsed if isinstance(message.parsed, schema) else schema.model_validate(message.parsed)
        except Exception as exc:
            record.update(status="ERROR", error=f"{type(exc).__name__}: {exc}")
            raise
        finally:
            record["seconds"] = round(fb_time.perf_counter() - started, 4)


## gpt-oss-20b fallback backend

Audit and repair requests use the local gpt-oss:20b model. The audit receives the complete registered input data (long tables are read page by page), and up to three repairs are allowed, including repeated repairs of the same layer.

In [ ]:
import ast
import copy
import csv
import difflib
import hashlib
import io
import json
import math
import re
import time
from collections import Counter, defaultdict
from datetime import date
from fractions import Fraction
from pathlib import Path

import requests

OSS_FALLBACK = {"model": "gpt-oss:20b", "think": "medium", "num_ctx": 65536, "temperature": 0.0, "seed": 42,
                "num_predict": {"artifact_audit": 8192, "repair": 16384, "data_chunk_audit": 2048},
                "http_timeout_seconds": 900, "max_format_retries": 1, "max_repairs": 3, "max_audits": 4,
                "context_reserve_tokens": 2048}
LITERAL_VIEW_THRESHOLD = 4096


class ModelFormatFailure(ValueError):
    """The model response is incomplete or not in the requested format."""


def canonical(value):
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"), default=str, allow_nan=False)


def _oss_write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, sort_keys=True, indent=2, default=str, allow_nan=False) + "\n",
                    encoding="utf-8")


LITERAL_VIEW_THRESHOLD = 4096


def _warning(warnings, code, **details):
    warnings.append({"code": code, **details})


def _event_rows(query, records, transformations, warnings):
    fields = {"revision", "effective_date", "action", "record_id"}
    events = [r for r in records if fields <= r["row"].keys()]
    if not events:
        return records
    explicit = (re.search(r"highest\s+integer\s+revision", query, re.I)
                and re.search(r"\bDELETE\b", query)
                and re.search(r"identical retransmissions count once", query, re.I))
    key_match = re.search(r"for each\s*\(([^)]+)\)", query, re.I)
    dates = set(re.findall(r"\b\d{4}-\d{2}-\d{2}\b", query))
    keys = [s.strip() for s in key_match[1].split(",")] if key_match else []
    tenants = [k for k in keys if k not in {"table", "record_id"}]
    if not explicit or len(events) != len(records) or len(dates) != 1 or len(tenants) != 1 or not {"table", "record_id"} <= set(keys):
        _warning(warnings, "EVENT_RULE_UNRESOLVED", action="Raw event rows retained")
        return records
    tenant_key = tenants[0]
    values = {r["row"].get(tenant_key, "") for r in events} - {""}
    mentioned = [v for v in values if re.search(r"(?<!\w)" + re.escape(v) + r"(?!\w)", query)]
    if len(mentioned) != 1:
        _warning(warnings, "EVENT_TENANT_UNRESOLVED", action="Raw event rows retained")
        return records
    asof, tenant = next(iter(dates)), mentioned[0]
    groups, excluded = defaultdict(list), []
    try:
        date.fromisoformat(asof)
        for record in events:
            row = record["row"]
            date.fromisoformat(row["effective_date"])
            if not set(keys) <= row.keys() or not re.fullmatch(r"[+-]?\d+", row["revision"]):
                raise ValueError("Missing event key or noninteger revision")
            if row["action"] not in {"UPSERT", "DELETE"}:
                raise ValueError("Unknown event action")
            if row[tenant_key] != tenant or row["effective_date"] > asof:
                excluded.append({"source_row": record["id"], "reason": "OUTSIDE_SCOPE_OR_FUTURE"})
            else:
                groups[tuple(row[k] for k in keys)].append(record)
        retained = set()
        for group in groups.values():
            unique = {}
            for record in group:
                signature = json.dumps(record["row"], sort_keys=True, ensure_ascii=False)
                if signature in unique:
                    excluded.append({"source_row": record["id"], "reason": "IDENTICAL_RETRANSMISSION"})
                else:
                    unique[signature] = record
            latest_revision = max(int(r["row"]["revision"]) for r in unique.values())
            latest = [r for r in unique.values() if int(r["row"]["revision"]) == latest_revision]
            if len(latest) != 1:
                raise ValueError("Conflicting records at the highest revision")
            chosen = latest[0]
            for record in unique.values():
                if record is not chosen or chosen["row"]["action"] == "DELETE":
                    excluded.append({"source_row": record["id"], "reason": "LATEST_DELETE" if record is chosen else "SUPERSEDED_REVISION"})
            if chosen["row"]["action"] != "DELETE":
                retained.add(chosen["id"])
    except (KeyError, TypeError, ValueError) as exc:
        _warning(warnings, "EVENT_VALUE_UNRESOLVED", error=str(exc), action="Raw event rows retained")
        return records
    selected = [r for r in records if r["id"] in retained]
    transformations.append({"kind": "EVENT_SELECTION", "key": keys, "tenant": tenant,
                            "as_of": asof, "input_rows": len(records), "retained_rows": len(selected),
                            "excluded": excluded})
    return selected


def _unit(value):
    value = value.lower()
    return {"minutes": "minute", "hours": "hour", "liters": "liter", "litres": "liter", "litre": "liter"}.get(value, value)


def _number(value):
    return str(value.numerator) if value.denominator == 1 else str(value)


def _normalize(query, records, transformations, warnings):
    records = copy.deepcopy(records)
    rules = defaultdict(set)
    for m in re.finditer(r"(\d+(?:\.\d+)?)\s+([A-Za-z]+)\s+per\s+([A-Za-z]+)", query):
        rules[_unit(m[3])].add((_unit(m[2]), Fraction(m[1]), m[0]))
    for record in records:
        row = record["row"]
        if not row.get("amount") or not row.get("unit"):
            continue
        choices = rules.get(_unit(row["unit"]), set())
        if len({(target, factor) for target, factor, _ in choices}) > 1:
            _warning(warnings, "CONFLICTING_UNIT_RULES", source_row=record["id"])
            continue
        if choices:
            target, factor, evidence = next(iter(choices))
            try:
                before = {"amount": row["amount"], "unit": row["unit"]}
                row.update(amount=_number(Fraction(row["amount"]) * factor), unit=target)
                transformations.append({"kind": "UNIT", "source_row": record["id"], "before": before,
                                        "after": {"amount": row["amount"], "unit": row["unit"]},
                                        "factor": _number(factor), "query_evidence": evidence})
            except (ValueError, ZeroDivisionError):
                _warning(warnings, "UNIT_VALUE_UNRESOLVED", source_row=record["id"])
    if re.search(r"amount\s*\*\s*usd_cents_numerator\s*/\s*denominator\s+using\s+fx", query, re.I):
        try:
            rates = defaultdict(set)
            for record in records:
                row = record["row"]
                if record["name"] == "fx":
                    rates[row["currency"]].add(Fraction(row["usd_cents_numerator"]) / Fraction(row["denominator"]))
            if not rates or any(len(v) != 1 for v in rates.values()):
                raise ValueError("Missing or conflicting FX rate")
            changes = []
            for record in records:
                row = record["row"]
                if record["name"] == "benefit" and row.get("currency") != "USD_CENTS":
                    rate = next(iter(rates[row["currency"]]))
                    changes.append((record, rate, _number(Fraction(row["amount"]) * rate)))
            for record, rate, amount in changes:
                row = record["row"]; before = {"amount": row["amount"], "currency": row["currency"]}
                row.update(amount=amount, currency="USD_CENTS")
                transformations.append({"kind": "FX", "source_row": record["id"], "before": before,
                                        "after": {"amount": amount, "currency": "USD_CENTS"}, "factor": _number(rate)})
        except (KeyError, ValueError, ZeroDivisionError, StopIteration) as exc:
            _warning(warnings, "FX_RULE_UNRESOLVED", error=str(exc), action="FX amounts retained")
    return records


def prepare_inputs(case, artifact_dir=None):
    """Read only registered CSVs; expose original and explicitly normalized views."""
    raw_tables, files, records, warnings, transformations = [], [], [], [], []
    seen = set()
    for index, item in enumerate(case["paths"]):
        path = Path(item["path"])
        if not path.is_absolute() or path.suffix.lower() != ".csv" or path in seen:
            raise ValueError("Registered inputs must be distinct absolute CSV paths")
        seen.add(path)
        raw = path.read_bytes(); digest = hashlib.sha256(raw).hexdigest()
        reader = csv.DictReader(io.StringIO(raw.decode("utf-8-sig"), newline=""), restkey="__extra_fields__")
        rows = list(reader); columns = list(reader.fieldnames or [])
        if len(set(columns)) != len(columns):
            raise ValueError("Duplicate CSV headers cannot be represented losslessly as dictionaries")
        if any("__extra_fields__" in row or any(v is None for v in row.values()) for row in rows):
            _warning(warnings, "RAGGED_CSV_ROWS", path=str(path))
        file_id = f"f{index:03d}"
        files.append({"id": file_id, "path": str(path), "sha256": digest, "bytes": len(raw), "row_count": len(rows)})
        raw_tables.append({"id": file_id, "name": path.stem, "view": "RAW_CSV", "path": str(path), "sha256": digest,
                           "columns": columns, "rows": rows})
        for offset, row in enumerate(rows):
            records.append({"id": f"{file_id}:r{offset}", "file_id": file_id,
                            "name": row.get("table") or path.stem, "row": copy.deepcopy(row)})
    selected = _event_rows(case["query"], records, transformations, warnings)
    normalized = _normalize(case["query"], selected, transformations, warnings)
    tables = {}
    for record in normalized:
        row, name = record["row"], record["name"]
        key = "table:" + name if row.get("table") else record["file_id"]
        table = tables.setdefault(key, {"id": key, "name": name, "view": "NORMALIZED_QUERY_VIEW", "columns": [], "rows": []})
        table["columns"].extend(k for k in row if k not in table["columns"])
        table["rows"].append(row)
    result = {"version": "prepared-inputs", "raw_tables": raw_tables, "tables": list(tables.values()),
              "transformations": transformations, "warnings": warnings, "files": files,
              "normalization": {"applied": bool(transformations), "raw_data_unchanged": True,
                                "instruction": "tables applies only the listed query transformations. Do not apply those conversions again; raw_tables keeps original values."},
              "number_representation": "CSV strings; exact conversions use integer or numerator/denominator strings. Normalized amounts are already converted once."}
    if artifact_dir is not None:
        path = Path(artifact_dir).resolve() / "prepared_inputs.json"
        result["data_path"] = str(path)
        _oss_write_json(path, result)
    return result


def _json_exact(value):
    if value is None or type(value) in (str, bool, int):
        return True
    if type(value) is float:
        return math.isfinite(value)
    if type(value) is list:
        return all(_json_exact(v) for v in value)
    return type(value) is dict and all(type(k) is str and _json_exact(v) for k, v in value.items())


def code_view(code, artifact_dir):
    """Externalize a large literal for the audit view only; unsupported code stays exact."""
    try:
        tree = ast.parse(code)
        matches = [n for n in tree.body if isinstance(n, (ast.Assign, ast.AnnAssign))
                   and len(n.targets if isinstance(n, ast.Assign) else [n.target]) == 1
                   and isinstance((n.targets if isinstance(n, ast.Assign) else [n.target])[0], ast.Name)
                   and (n.targets if isinstance(n, ast.Assign) else [n.target])[0].id == "CSVQA_DATA"]
        if len(matches) != 1:
            return code
        node = matches[0]
        if len(ast.get_source_segment(code, node.value) or "") < LITERAL_VIEW_THRESHOLD:
            return code
        value = ast.literal_eval(node.value)
        if not _json_exact(value):
            return code
        raw = (json.dumps(value, ensure_ascii=False, allow_nan=False) + "\n").encode("utf-8")
        path = Path(artifact_dir).resolve() / ("csvqa_literal_" + hashlib.sha256(raw).hexdigest() + ".json")
        path.parent.mkdir(parents=True, exist_ok=True); path.write_bytes(raw)
        lines = code.splitlines(keepends=True)
        start = sum(len(s.encode("utf-8")) for s in lines[:node.lineno - 1]) + node.col_offset
        end = sum(len(s.encode("utf-8")) for s in lines[:node.end_lineno - 1]) + node.end_col_offset
        replacement = f"import pandas as pd\nCSVQA_DATA = pd.read_json({str(path)!r}, typ='series').to_dict()"
        view = (code.encode("utf-8")[:start] + replacement.encode("utf-8") + code.encode("utf-8")[end:]).decode("utf-8")
        return view
    except (SyntaxError, ValueError, TypeError, RecursionError, OSError):
        return code


AUDIT_SYSTEM = """Audit the supplied optimization problem, formulation, complete program and runtime.
Check Logic, then Data, then Code. Logic: objective, variables and every stated constraint;
Data: all files, rows, joins, units, versions and exclusions; Code: implementation and execution.
A finite OPTIMAL solve does not establish that the formulation is correct. Use UNKNOWN when unsure.
Return only JSON with logic, data, code. Each has status PASS/FAIL/UNKNOWN and short issue,
explanation, fix strings. No citations, clause IDs or proof objects are required.
Data-page notes are reading notes, not a prior verdict. Files remain completely available."""


REPAIR_SYSTEM = """Repair the complete optimization program for the supplied query and audit/runtime errors.
Return one complete Python program, raw or in a python code fence, not a patch or JSON.
For a logic repair, first give a short updated formulation prefixed FORMULATION:.
If FORMULATION: is present, follow it with a CODE: line or a fenced Python block. Otherwise return only raw complete code.
For data/code repairs retain the supplied formulation unless explicitly updating it with FORMULATION:.
Use gurobipy, expose the model as m or model, and do not dispose it before the executor inspects it.
All prepared tables have rows as dictionaries keyed by their columns. Complete prepared data can be read
by assigning DATA_PATH to the exact string in data.data_path, then using
data = pd.read_json(DATA_PATH, typ='series'); tables = {t['name']: t['rows'] for t in data['tables']}.
Use pd.read_csv for registered original CSVs. The isolated executor forbids open, network and subprocess calls.
Preserve unrelated correct source and change the problem-causing parts; return the full program.
Respect the displayed transformations; do not convert units/versions twice. Preserve legitimate fractions.
Do not invent constraints or silently drop rows. Syntax/type/runtime errors below are actual candidate feedback."""


PAGE_SYSTEM = """Read this complete data page for the supplied problem. It is only a row range, not a whole table.
Give a short factual reading note (at most 300 characters) about fields, values, joins, units or versions.
Do not fail a page because other registered pages contain the remaining rows; defer global questions.
Return JSON with one string field note. No verdict, references or formal proof is requested."""


LAYER_SCHEMA = {"type": "object", "additionalProperties": False,
    "properties": {"status": {"type": "string", "enum": ["PASS", "FAIL", "UNKNOWN"]},
        **{k: {"type": "string"} for k in ("issue", "explanation", "fix")}},
    "required": ["status", "issue", "explanation", "fix"]}


AUDIT_SCHEMA = {"type": "object", "additionalProperties": False,
    "properties": {k: copy.deepcopy(LAYER_SCHEMA) for k in ("logic", "data", "code")},
    "required": ["logic", "data", "code"]}


PAGE_SCHEMA = {"type": "object", "additionalProperties": False,
    "properties": {"note": {"type": "string", "maxLength": 300}}, "required": ["note"]}


def finite_optimal(runtime):
    obj = runtime.get("objective")
    return (runtime.get("status") == "OPTIMAL" and not runtime.get("error")
        and type(obj) in (int, float) and math.isfinite(obj))


def unknown_audit(error):
    return {k: {"status": "UNKNOWN", "issue": "Audit response unavailable",
        "explanation": str(error), "fix": "Check the full problem, data and execution; provide a corrected complete program."}
        for k in ("logic", "data", "code")}


def repair_text(raw, current_formulation):
    """Extract a plain complete program, without syntax/type rewriting or checking."""
    match = re.search(r"```(?:python|py)?\s*\n(.*?)```", raw, re.S | re.I)
    prefix = raw[:match.start()] if match else ""
    code = match.group(1).strip("\n") if match else raw.strip()
    formulation = current_formulation
    if match and "FORMULATION:" in prefix:
        formulation = prefix.split("FORMULATION:", 1)[1].strip()
    elif raw.lstrip().startswith("FORMULATION:") and "\nCODE:" in raw:
        before, code = raw.split("\nCODE:", 1)
        formulation, code = before.split("FORMULATION:", 1)[1].strip(), code.strip()
    return formulation, code


def _table_views(prepared):
    tables = [{"view": "prepared", **t} for t in prepared["tables"]]
    for raw in prepared.get("raw_tables", []):
        if not any(t.get("id") == raw.get("id") and t.get("name") == raw.get("name") and t.get("columns") == raw.get("columns")
                   and t.get("rows") == raw.get("rows") for t in prepared["tables"]):
            tables.append({"view": "raw", **raw})
    return tables


def _summary(prepared):
    rules = {}; counts = Counter()
    for change in prepared.get("transformations", []):
        kind = change.get("kind", "OTHER"); counts[kind] += 1
        if kind == "EVENT_SELECTION":
            rule = {k: v for k, v in change.items() if k != "excluded"}
            rule["excluded_count"] = len(change.get("excluded", []))
            rule["excluded_by_reason"] = dict(Counter(row.get("reason", "UNKNOWN") for row in change.get("excluded", [])))
        else:
            rule = {"kind": kind, "factor": change.get("factor"),
                "from": {k: v for k, v in change.get("before", {}).items() if k != "amount"},
                "to": {k: v for k, v in change.get("after", {}).items() if k != "amount"}}
        rules[canonical(rule)] = rule
    return {"data_path": prepared.get("data_path"), "files": [{"name": Path(f["path"]).name, "path": f["path"]} for f in prepared.get("files", [])],
        "transformations": {"rules": list(rules.values()), "counts": dict(counts), "complete_records": prepared.get("data_path")},
        "warnings": dict(Counter(w.get("code", "WARNING") for w in prepared.get("warnings", []))),
        "normalization": prepared.get("normalization", {}), "number_representation": prepared.get("number_representation"),
        "tables": [{k: t.get(k) for k in ("name", "columns")} | {"row_count": len(t["rows"])} for t in prepared["tables"]]}


def _pages(query, tables, target_bytes=32000):
    fragments = []
    for ti, table in enumerate(tables):
        rows = table["rows"]; batch = []; start = 0
        base = {k: v for k, v in table.items() if k != "rows"} | {"table_index": ti, "total_rows": len(rows)}
        for index, row in enumerate(rows):
            if batch and len(canonical({**base, "rows": [*batch, row]}).encode()) > target_bytes:
                fragments.append({**base, "rows": batch, "row_start": start, "row_count": len(batch)})
                start, batch = index, []
            batch.append(row)
        if batch or not rows: fragments.append({**base, "rows": batch, "row_start": start, "row_count": len(batch)})
    pages, current = [], []
    for fragment in fragments:
        if current and len(canonical([*current, fragment]).encode()) > target_bytes:
            pages.append({"query": query, "tables": current}); current = []
        current.append(fragment)
    if current: pages.append({"query": query, "tables": current})
    return pages


def _raw_coverage(prepared, shown):
    return all(any(view.get("id") == raw.get("id") and view["columns"] == raw["columns"] and view["rows"] == raw["rows"]
        for view in shown) for raw in prepared.get("raw_tables", []))


def _page_coverage(tables, pages):
    restored = [[] for _ in tables]; starts = [0 for _ in tables]
    for page in pages:
        for fragment in page["tables"]:
            index = fragment["table_index"]
            if fragment["columns"] != tables[index]["columns"] or fragment["row_start"] != starts[index]: return False
            restored[index].extend(fragment["rows"]); starts[index] += len(fragment["rows"])
    return all(rows == table["rows"] for rows, table in zip(restored, tables))


def _candidate_view(candidate, directory):
    return {"formulation": candidate.get("formulation", ""),
        "code": code_view(candidate.get("code", ""), directory),
        "runtime": {k: candidate.get("runtime", {}).get(k) for k in ("status", "objective")} | {
            "error": str(candidate["runtime"]["error"])[-1200:] if candidate.get("runtime", {}).get("error") else None,
            "log_tail": (candidate.get("runtime", {}).get("log_tail") or "")[-1200:]}}


def _audit_payload(case, candidate, prepared, notes, directory, paged):
    data = _summary(prepared)
    if not paged: data["tables"] = _table_views(prepared)
    payload = {"query": case["query"], "candidate": _candidate_view(candidate, directory), "data": data}
    if paged: payload["data_page_notes"] = notes
    return payload


def request_body(kind, system, payload, schema=None, retry_error=None):
    if kind not in OSS_FALLBACK["num_predict"]: raise ValueError("Unknown simple request kind")
    if retry_error is not None:
        system += "\nSingle format recovery: previous response was not in the requested format. " + str(retry_error)[:600] + "\nRepeat the unchanged task; do not invent facts."
    body = {"model": OSS_FALLBACK["model"], "messages": [{"role": "system", "content": system},
        {"role": "user", "content": json.dumps(payload, ensure_ascii=False, sort_keys=True, separators=(",", ":"), default=str)}],
        "think": OSS_FALLBACK["think"], "stream": False, "options": {"num_ctx": OSS_FALLBACK["num_ctx"],
            "num_predict": OSS_FALLBACK["num_predict"][kind], "temperature": OSS_FALLBACK["temperature"], "seed": OSS_FALLBACK["seed"]}}
    if schema is not None: body["format"] = copy.deepcopy(schema)
    return body


def input_budget(body):
    size = len(json.dumps(body, ensure_ascii=False, sort_keys=True, separators=(",", ":"), default=str).encode())
    limit = body["options"]["num_ctx"] - body["options"]["num_predict"] - OSS_FALLBACK["context_reserve_tokens"]
    return {"input_bytes": size, "input_limit": limit, "within_budget": size <= limit,
            "measurement": "complete_request_utf8_bytes_conservative_token_upper_bound"}


def _parse(kind, content):
    if not isinstance(content, str) or not content.strip(): raise ModelFormatFailure("Empty model response")
    if kind == "repair": return content
    try: value = json.loads(content)
    except json.JSONDecodeError as exc: raise ModelFormatFailure(str(exc)) from exc
    if kind == "data_chunk_audit":
        if not isinstance(value, dict) or set(value) != {"note"} or not isinstance(value["note"], str) or len(value["note"]) > 300:
            raise ModelFormatFailure("Page response requires one note string of at most 300 characters")
        return value
    if not isinstance(value, dict) or set(value) != {"logic", "data", "code"}:
        raise ModelFormatFailure("Audit requires exactly logic, data, code")
    for layer in value.values():
        if (not isinstance(layer, dict) or set(layer) != {"status", "issue", "explanation", "fix"}
            or layer["status"] not in {"PASS", "FAIL", "UNKNOWN"}
            or any(not isinstance(layer[k], str) for k in ("issue", "explanation", "fix"))):
            raise ModelFormatFailure("Each layer needs status PASS/FAIL/UNKNOWN and issue/explanation/fix strings")
    return value


def prepare_plan(case, primary, artifact_dir=None):
    """Pure input/request planning; no client, embedding or solver invocation."""
    directory = Path(artifact_dir) if artifact_dir is not None else None
    prepared = prepare_inputs(case, artifact_dir=directory)
    original = {"formulation": primary.get("generated_model", ""), "code": primary.get("solve_code", ""),
        "runtime": {"status": "OPTIMAL" if primary.get("final_ok") is True else "FAILED",
            "objective": primary.get("final_objective"), "error": primary.get("primary_error"),
            "log_tail": primary.get("primary_log_tail", "")}}
    view = _candidate_view(original, directory)
    summary = _summary(prepared)
    payload = {"query": case["query"], "candidate": view, "data": {**summary, "tables": _table_views(prepared)}}
    initial = request_body("artifact_audit", AUDIT_SYSTEM, payload, AUDIT_SCHEMA)
    if input_budget(initial)["within_budget"]:
        # Use the repair budget as well: otherwise an audit can fit while no repair can run.
        preview = _plan_result(prepared, [], payload, [{"kind": "artifact_audit", "body": initial, **input_budget(initial)}], initial)
        if all(r["within_budget"] for r in preview["requests"]): return preview
    pages = _pages(case["query"], _table_views(prepared))
    payload["data"] = summary
    payload["data_page_notes"] = [{"page": i, "note": "~" * 300} for i in range(len(pages))]
    body = request_body("artifact_audit", AUDIT_SYSTEM, payload, AUDIT_SCHEMA)
    requests = [{"kind": "data_chunk_audit", "body": (b := request_body("data_chunk_audit", PAGE_SYSTEM, page, PAGE_SCHEMA)),
                 **input_budget(b)} for page in pages]
    requests.append({"kind": "artifact_audit", "body": body, "template": True, **input_budget(body)})
    return _plan_result(prepared, pages, payload, requests, body)


def repair_payload(payload, audit, layer):
    full = {name: copy.deepcopy(audit[name]) for name in ("logic", "data", "code")}
    result = {**payload, "repair_layer": layer, "audit": full}
    if not input_budget(request_body("repair", REPAIR_SYSTEM, result))["within_budget"]:
        # Keep issues/fixes verbatim; only an overlong explanatory passage is abbreviated.
        result["audit"] = {name: {**row, "explanation": row["explanation"][:250] + " ... " + row["explanation"][-250:]
            if len(row["explanation"]) > 500 else row["explanation"]} for name, row in full.items()}
        result["audit_note"] = "Explanations abbreviated because the full request exceeded context; complete raw audit is preserved in the per-round audit artifact. All issue/fix text is unchanged."
    return result


def _plan_result(prepared, pages, payload, requests, initial):
    template_audit = {name: {"status": "UNKNOWN", "issue": "~"*300, "explanation": "~"*1000, "fix": "~"*1200}
                      for name in ("logic", "data", "code")}
    repair = request_body("repair", REPAIR_SYSTEM, repair_payload(payload, template_audit, "logic"))
    requests.append({"kind": "repair", "body": repair, "template": True,
        "template_reason": "FIRST_REPAIR_IF_INITIAL_AUDIT_NOT_VALIDATED_WITH_WIDE_FULL_AUDIT_NOTES", **input_budget(repair)})
    return {"prepared": prepared, "pages": pages, "initial_payload": payload, "initial_body": initial,
        "requests": requests, "raw_coverage_complete": _raw_coverage(prepared, _table_views(prepared))
            and (not pages or _page_coverage(_table_views(prepared), pages))}


class OssFallbackService:
    """Audit, data-page and repair requests to the local gpt-oss:20b model."""

    def __init__(self, endpoint=None):
        self.endpoint = endpoint or OLLAMA_BASE_URL.rstrip("/") + "/api/chat"
        self.calls = []

    def _ask(self, kind, system, payload, schema=None, layer=""):
        retry_error = None
        for attempt in range(OSS_FALLBACK["max_format_retries"] + 1):
            body = request_body(kind, system, payload, schema, retry_error)
            if not input_budget(body)["within_budget"]:
                raise ModelFormatFailure("Complete request exceeds context budget; no data was truncated")
            started = time.perf_counter()
            record = {"kind": kind, "layer": layer, "attempt": attempt, "status": "STARTED"}
            self.calls.append(record)
            try:
                response = requests.post(self.endpoint, json=body, timeout=OSS_FALLBACK["http_timeout_seconds"])
                response.raise_for_status()
                data = response.json()
                record.update(input_tokens=data.get("prompt_eval_count"), output_tokens=data.get("eval_count"),
                              finish_reason=data.get("done_reason"))
                if data.get("done") is not True or data.get("done_reason") != "stop":
                    record["status"] = "TRUNCATED"
                    raise ModelFormatFailure("Incomplete model output: " + str(data.get("done_reason")))
                parsed = _parse(kind, data["message"].get("content"))
                record["status"] = "COMPLETED"
                return parsed
            except ModelFormatFailure as exc:
                if record["status"] == "STARTED":
                    record["status"] = "INVALID_SCHEMA"
                record["error"] = str(exc)
                retry_error = str(exc)
                if attempt == OSS_FALLBACK["max_format_retries"]:
                    raise
            except BaseException as exc:
                record.update(status="ERROR", error=f"{type(exc).__name__}: {exc}")
                raise
            finally:
                record["seconds"] = round(time.perf_counter() - started, 4)

    def audit(self, payload):
        return self._ask("artifact_audit", AUDIT_SYSTEM, payload, AUDIT_SCHEMA)

    def page(self, payload):
        return self._ask("data_chunk_audit", PAGE_SYSTEM, payload, PAGE_SCHEMA)

    def repair(self, payload, layer="code"):
        return self._ask("repair", REPAIR_SYSTEM, payload, layer=layer)


def oss_fallback_case(case, primary, work_root, service=None):
    """Audit the Primary output, repair up to three times, re-execute and re-audit."""
    began = time.perf_counter(); original_primary = copy.deepcopy(primary)
    directory = Path(work_root) / "cases" / case["case_id"]
    directory.mkdir(parents=True, exist_ok=True)
    _oss_write_json(directory / "primary.json", original_primary)
    service = service or OssFallbackService()
    plan = prepare_plan(case, primary, directory)
    prepared, pages = plan["prepared"], plan["pages"]
    _oss_write_json(directory / "input_plan.json", {"pages": len(pages), "data_path": prepared.get("data_path"),
        "requests": [{k: v for k, v in r.items() if k != "body"} for r in plan["requests"]]})
    original = copy.deepcopy(fb_candidate(primary))
    if not finite_optimal(original["runtime"]):
        original["runtime"] = {**original["runtime"], "status": "FAILED", "objective": None,
            "error": original["runtime"].get("error") or "PRIMARY_NOT_FINITE_OPTIMAL"}
    candidate = copy.deepcopy(original); audits = []; raw_audits = []; repairs = []; notes = []; warning = None; validated = False
    for index, page in enumerate(pages):
        try: note = service.page(page)["note"]
        except ModelFormatFailure as exc: note = "Page reading unavailable: " + str(exc)
        notes.append({"page": index, "note": note})
        _oss_write_json(directory / f"page_{index:03d}.json", {"input": page, "note": note})
    for turn in range(4):
        payload = _audit_payload(case, candidate, prepared, notes, directory, bool(pages))
        try:
            verdict = service.audit(payload)
            error = None
        except ModelFormatFailure as exc:
            verdict, error = unknown_audit(exc), str(exc)
        raw_audits.append(copy.deepcopy(verdict))
        audit = copy.deepcopy(verdict)
        if all(verdict[k]["status"] == "PASS" for k in ("logic", "data", "code")) and not finite_optimal(candidate["runtime"]):
            audit["code"] = {"status": "FAIL", "issue": "Candidate execution failed", "explanation": str(candidate["runtime"]), "fix": "Correct the actual execution error."}
        audit.update(index=turn, format_error=error, first_failed_layer=next(
            (k for k in ("logic", "data", "code") if audit[k]["status"] != "PASS"), "none"))
        audits.append(audit); _oss_write_json(directory / f"audit_{turn:02d}.json", audit)
        all_pass = all(verdict[k]["status"] == "PASS" for k in ("logic", "data", "code"))
        if all_pass and finite_optimal(candidate["runtime"]): validated = True; break
        if len(repairs) >= 3: break
        layer = next((k for k in ("logic", "data", "code") if verdict[k]["status"] != "PASS"), "code")
        request = repair_payload(payload, verdict, layer)
        before = copy.deepcopy(candidate)
        try:
            raw = service.repair(request, layer=layer)
            formulation, code = repair_text(raw, candidate["formulation"])
            execution_attempted = True
            executor_error_origin = None
            try:
                runtime = fb_execute_fallback(code,
                    [p["path"] for p in case["paths"]] + ([prepared["data_path"]] if prepared.get("data_path") else []), timeout=1800)
            except SyntaxError as exc:
                runtime = {"status": "FAILED", "objective": None,
                    "error": f"{type(exc).__name__}: {exc}", "error_type": type(exc).__name__, "log_tail": "", "failure_phase": "candidate_source_parse"}
                executor_error_origin = "native_source_extractor"
            summary = "Complete program returned for " + layer + " repair"
        except ModelFormatFailure as exc:
            execution_attempted = False
            executor_error_origin = "model_response_format"
            raw, code, formulation = "", candidate["code"], candidate["formulation"]
            runtime = {"status": "FAILED", "objective": None, "error": "REPAIR_FORMAT_FAILED:" + str(exc), "log_tail": ""}
            summary = "Model repair format failed; retained source, execution feedback supplied to next audit"
        candidate = {**candidate, "formulation": formulation, "code": code, "runtime": runtime}
        repair = {"index": len(repairs), "layer": layer, "summary": summary, "formulation": formulation,
            "code": code, "code_before": before["code"], "code_diff": "".join(difflib.unified_diff(
                before["code"].splitlines(True), code.splitlines(True), fromfile="before.py", tofile="after.py")),
            "raw_response": raw, "runtime": runtime, "execution_attempted": execution_attempted,
            "executor_error_origin": executor_error_origin, "global_reaudit_required": True}
        repairs.append(repair)
        (directory / f"candidate_{len(repairs):02d}.py").write_text(code, encoding="utf-8")
        _oss_write_json(directory / f"repair_{len(repairs):02d}_{layer}.json", repair)
        _oss_write_json(directory / f"candidate_{len(repairs):02d}.json", candidate)
    selected = copy.deepcopy(candidate if validated else original)
    if validated:
        source, trust = ("COMPOSITE", "FALLBACK_VALIDATED") if repairs else ("PRIMARY", "PRIMARY_VALIDATED")
    elif finite_optimal(original["runtime"]):
        source, trust = "PRIMARY", "PRIMARY_UNVERIFIED"
    else:
        source, trust = "NONE", "FAILED"
        selected["runtime"] = {**selected["runtime"], "status": "FAILED", "objective": None,
            "error": selected["runtime"].get("error") or "NO_VALIDATED_FINITE_CANDIDATE"}
    if not validated: warning = "FALLBACK_NOT_VALIDATED_AFTER_AUDIT_AND_REPAIR"
    result = {"case_id": case["case_id"], "paths": copy.deepcopy(case["paths"]), "primary": original_primary,
        "candidate": candidate, "selected": {**selected, "source": source, "trust": trust},
        "audits": audits, "raw_audits": raw_audits, "repairs": repairs, "data_page_notes": notes,
        "input_summary": _summary(prepared), "fallback_calls": copy.deepcopy(getattr(service, "calls", [])),
        "warning": warning,
        "elapsed_seconds": round(time.perf_counter() - began, 6)}
    _oss_write_json(directory / "final_result.json", result)
    return result
